In [ ]:
import pickle
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader, TensorDataset
from sklearn.metrics import f1_score, mean_squared_error, r2_score
from scipy.stats import spearmanr
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix
import os
import matplotlib.pyplot as plt
from tqdm import tqdm
import matplotlib.patches as mpatches
from matplotlib.colors import ListedColormap
from sklearn.metrics import ndcg_score
import math


In [ ]:

# --- CONFIGURATION ---
class Config:
    # UPDATE THIS PATH TO YOUR FILE
    data_path = 'reduced_256_embedd.pkl'

    epochs_trans = 1000
    epochs_cis = 100
    batch_size = 512

    # --- BINS INFO ---
    interval = [-np.inf, -.5, 1., np.inf]
    inter_names = ['Deleterious', 'Neutral', 'Beneficial']
    label_col = ['#8B0000', "#0AEE0A", '#00008B'] #  '#FF4500',, '#00BFFF'

    # --- DATA LIMITS ---
    max_samples = None  # Set to None to use all
    max_seq_len = 256
    # -------------------

    # Sequence encoder params
    seq_hidden = 256
    seq_layers = 1
    seq_dropout = 0.1

    # Hyperbolic Params
    hyp_lr = 5e-5
    hyp_dim = 1024
    n_head = 8

    # Euclidean Params
    euc_lr = 5e-4
    euc_hidden = [1024, 512]

    output_dim = 3
    val_frac = 0.15
    seed = 80085
    device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# --- UTILS ---

def discretize_fitness(fitness_array):
    """Transforms continuous fitness into 4 discrete classes for F1 comparison."""
    bins = Config.interval
    labels = np.digitize(fitness_array, bins) - 1
    return labels



def visualize_fitness_distribution_colored(model, loader, device, model_type="euclidean"):
    model.eval()
    all_preds = []
    all_true = []

    # 1. Inference
    with torch.no_grad():
        for feats, fitness in loader:
            feats = feats.to(device)
            fitness = fitness.to(device)

            if model_type == "euclidean":
                preds = model(feats.float())
            else:
                preds = model(feats.double())

            all_preds.append(preds.cpu().numpy())
            all_true.append(fitness.cpu().numpy())

    y_pred_raw = np.concatenate(all_preds)
    y_true_raw = np.concatenate(all_true)

    # 2. Discretizza le predizioni
    pred_classes = discretize_fitness(y_pred_raw)

    # 3. Ordina i dati in base alla Fitness REALE
    data = sorted(zip(y_true_raw, pred_classes), key=lambda x: x[0])
    y_true_sorted, pred_cls_sorted = zip(*data)

    # --- MODIFICA PER DISPERDERE I PUNTI ---
    n_samples = len(y_true_sorted)

    # Creiamo un asse X base
    x_base = np.arange(n_samples)

    # Aggiungiamo "Jitter" (rumore) per allargare la linea in una nuvola
    # Il rumore è proporzionale al numero di campioni (es. 2% della larghezza totale)
    jitter_strength = n_samples * 0.02
    noise = np.random.normal(0, jitter_strength, size=n_samples)
    x_axis_dispersed = x_base + noise
    # ---------------------------------------

    # 4. Configurazione Colori
    colors = Config.label_col
    cmap = ListedColormap(colors)
    class_labels =  Config.inter_names

    plt.figure(figsize=(12, 7)) # Figura un po' più grande

    # 5. Scatter Plot "Disperso"
    # s=25: punti più piccoli per vedere meglio la densità
    # alpha=0.7: trasparenza per vedere le sovrapposizioni
    scatter = plt.scatter(x_axis_dispersed, y_true_sorted, c=pred_cls_sorted, cmap=cmap,
                          vmin=0, vmax=4, alpha=0.7, s=25, edgecolors='none')

    plt.axhline(0, color='black', linestyle='--', alpha=0.3, label='Baseline')

    # 6. Legenda
    patches = [mpatches.Patch(color=colors[i], label=class_labels[i]) for i in range(len(class_labels))]
    plt.legend(handles=patches, title="Classe Predetta", loc='upper left')

    plt.xlabel('Sample Index (Sorted by Fitness) + Jitter')
    plt.ylabel('Real Fitness Score')
    plt.title(f'Dispersed Fitness Distribution ({model_type})')
    plt.grid(True, alpha=0.15)

    # Assicura che i limiti dell'asse X siano puliti nonostante il jitter
    plt.xlim(-jitter_strength*3, n_samples + jitter_strength*3)

    plt.show()

class MutationDataset(Dataset):
    def __init__(self, pkl_path, max_samples=None, max_seq_len=None):
        print(f"Loading {pkl_path}...")
        with open(pkl_path, 'rb') as f:
            data = pickle.load(f)

        if hasattr(data, 'loc'):
            self.fitness = data['fitness'].values
            embeddings_raw = data['embedding'].values
            self.file_names = data['file_name'].values
        else:
            self.fitness = np.array(data['fitness'])
            embeddings_raw = np.array(data['embedding'])
            self.file_names = np.array(data['file_name'])

        if max_samples is not None and max_samples < len(self.fitness):
            print(f"Limiting dataset to first {max_samples} samples.")
            self.fitness = self.fitness[:max_samples]
            embeddings_raw = embeddings_raw[:max_samples]
            self.file_names = self.file_names[:max_samples]

        self.labels = discretize_fitness(self.fitness)

        np_embs = [np.array(e) for e in embeddings_raw]

        max_h_data = max(s[0] for s in [e.shape for e in np_embs])
        if max_seq_len:
            self.max_h = min(max_h_data, max_seq_len)
        else:
            self.max_h = max_h_data

        if len(np_embs[0].shape) > 1:
            self.max_w = max(s[1] for s in [e.shape for e in np_embs])
        else:
            self.max_w = 1

        print(f"Processing Embeddings: Target Shape=({self.max_h}, {self.max_w})")

        processed = []
        lengths = []
        for arr in tqdm(np_embs, desc="Padding/Truncating"):
            if len(arr.shape) == 1:
                current_len = arr.shape[0]
                limit = min(current_len, self.max_h)
                padded = np.zeros((self.max_h, 1), dtype=np.float32)
                padded[:limit, 0] = arr[:limit]
                lengths.append(limit)
            else:
                h, w = arr.shape
                limit_h = min(h, self.max_h)
                limit_w = min(w, self.max_w)
                padded = np.zeros((self.max_h, self.max_w), dtype=np.float32)
                padded[:limit_h, :limit_w] = arr[:limit_h, :limit_w]
                lengths.append(limit_h)
            processed.append(padded)

        self.embeddings = np.stack(processed, axis=0)
        self.lengths = np.array(lengths, dtype=np.int32)

        mean = self.embeddings.mean(axis=(0, 1))
        std = self.embeddings.std(axis=(0, 1)) + 1e-6
        self.embeddings = (self.embeddings - mean[None, None, :]) / std[None, None, :]

        self.input_dim = self.max_w
        self.proteins = np.array([str(fn).split('_')[0] for fn in self.file_names])

    def __len__(self):
        return len(self.embeddings)

    def __getitem__(self, idx):
        return {
            'embedding': torch.tensor(self.embeddings[idx], dtype=torch.float32),
            'length': int(self.lengths[idx]),
            'fitness': torch.tensor(self.fitness[idx], dtype=torch.float32),
            'label': torch.tensor(self.labels[idx], dtype=torch.long),
            'protein': self.proteins[idx]
        }

def visualize_confusion_matrix(model, loader, device, model_type="euclidean", title="Confusion Matrix"):
    model.eval()
    all_preds = []
    all_true = []

    # 1. Raccogli tutte le predizioni
    with torch.no_grad():
        for feats, fitness in loader:
            feats = feats.to(device)
            fitness = fitness.to(device)

            if model_type == "euclidean":
                preds = model(feats.float())
            else:
                preds = model(feats.double())

            all_preds.append(preds.cpu().numpy())
            all_true.append(fitness.cpu().numpy())

    y_pred_raw = np.concatenate(all_preds)
    y_true_raw = np.concatenate(all_true)

    # 2. Discretizza (trasforma valori in classi 0-4)
    y_pred_cls = discretize_fitness(y_pred_raw)
    y_true_cls = discretize_fitness(y_true_raw)

    # 3. Calcola la Matrice
    cm = confusion_matrix(y_true_cls, y_pred_cls)

    # 4. Plotting con Seaborn
    plt.figure(figsize=(8, 6))

    # Definiamo etichette leggibili in base ai tuoi bin
    # Bins: [-inf, -3, -1, 1, 3, inf] -> 5 Classi
    class_names = Config.inter_names

    # Assicuriamoci che le etichette corrispondano alle classi presenti
    # (Se nel validation set mancano alcune classi, questo evita errori)
    unique_classes = sorted(list(set(y_true_cls) | set(y_pred_cls)))
    labels = [class_names[i] for i in unique_classes if i < len(class_names)]

    sns.heatmap(cm, annot=True, fmt='d', cmap='Blues',
                xticklabels=labels, yticklabels=labels)

    plt.xlabel('Predicted Label')
    plt.ylabel('True Label')
    plt.title(f'{title} (Model: {model_type})')
    plt.show()

def collate_fn(batch):
    embs = torch.stack([b['embedding'] for b in batch])
    lengths = torch.tensor([b['length'] for b in batch], dtype=torch.long)
    fitness = torch.stack([b['fitness'] for b in batch])
    labels = torch.stack([b['label'] for b in batch])
    proteins = [b['protein'] for b in batch]
    return embs, lengths, fitness, labels, proteins





# --- Modulo Helper per la Positional Encoding ---
class PositionalEncoding(nn.Module):
    def __init__(self, d_model, max_len=5000, dropout=0.1):
        super().__init__()
        self.dropout = nn.Dropout(p=dropout)

        # Crea la matrice di encoding una volta sola nel costruttore
        pe = torch.zeros(max_len, d_model)
        position = torch.arange(0, max_len, dtype=torch.float).unsqueeze(1)
        div_term = torch.exp(torch.arange(0, d_model, 2).float() * (-math.log(10000.0) / d_model))

        pe[:, 0::2] = torch.sin(position * div_term)
        pe[:, 1::2] = torch.cos(position * div_term)

        pe = pe.unsqueeze(0) # (1, max_len, d_model)
        self.register_buffer('pe', pe)

    def forward(self, x):
        # x: (Batch, Seq_Len, Dim)
        # Aggiunge il segnale posizionale all'embedding
        x = x + self.pe[:, :x.size(1), :]
        return self.dropout(x)

# ==========================================
# Sequence encoder: Transformer + Attention pooling
# ==========================================
class SequenceEncoder(nn.Module):
    def __init__(self, input_dim, hidden_size=256, num_layers=2, dropout=0.1, nhead=8):
        super().__init__()

        # Controllo di sicurezza
        assert hidden_size % nhead == 0, \
            f"Errore: hidden_size ({hidden_size}) deve essere divisibile per nhead ({nhead})"

        self.input_dim = input_dim
        self.hidden_size = hidden_size

        # 1. Proiezione Iniziale
        self.input_proj = nn.Linear(input_dim, hidden_size)

        # 2. Positional Encoding
        self.pos_encoder = PositionalEncoding(hidden_size, dropout=dropout)

        # 3. Transformer Encoder Layer
        # RIMOSSO enable_nested_tensor=False perché la tua versione di PyTorch è vecchia
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=nhead,
            dim_feedforward=hidden_size * 4,
            dropout=dropout,
            batch_first=True
        )
        self.transformer = nn.TransformerEncoder(encoder_layer, num_layers=num_layers)

        # 4. Attention Pooling Head
        self.out_dim = hidden_size
        self.attn_proj = nn.Linear(self.out_dim, self.out_dim // 2)
        self.attn_vec = nn.Linear(self.out_dim // 2, 1)

    def forward(self, x, lengths):
        x = x.float()
        device = x.device
        batch_size, seq_len, _ = x.shape

        # A. Maschera di padding
        padding_mask = torch.arange(seq_len, device=device).unsqueeze(0) >= lengths.unsqueeze(1)

        # B. Proiezione + Positional Encoding
        x = self.input_proj(x)
        x = self.pos_encoder(x)

        # C. Transformer
        out = self.transformer(x, src_key_padding_mask=padding_mask)

        # D. Attention Pooling
        proj = torch.tanh(self.attn_proj(out))
        scores = self.attn_vec(proj).squeeze(-1)

        scores = scores.masked_fill(padding_mask, float('-inf'))

        attn = torch.softmax(scores, dim=-1)
        attn = attn.unsqueeze(-1)

        pooled = torch.sum(out * attn, dim=1)

        return pooled




# ==========================================
# Hyperbolic utils & regressors
# ==========================================
class HyperbolicUtils:
    EPS = 1e-5

    @staticmethod
    def artanh(x):
        x = torch.clamp(x, min=-1.0 + 1e-5, max=1.0 - 1e-5)
        return 0.5 * torch.log((1 + x) / (1 - x))

    @staticmethod
    def exp_map_zero(v):
        v_norm = torch.norm(v, p=2, dim=-1, keepdim=True).clamp_min(1e-6)
        scaled = torch.tanh(v_norm) * (v / v_norm)
        return scaled

    @staticmethod
    def log_map_zero(y):
        y_norm = torch.norm(y, p=2, dim=-1, keepdim=True).clamp(min=1e-6, max=1.0 - 1e-5)
        scale = HyperbolicUtils.artanh(y_norm) / y_norm
        return y * scale


class HyperbolicRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims, hyp_dim):
        super().__init__()
        layers = []
        in_d = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_d, h_dim))
            layers.append(nn.LayerNorm(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            in_d = h_dim
        self.encoder = nn.Sequential(*layers)
        self.to_tangent = nn.Linear(in_d, hyp_dim)
        self.regressor = nn.Linear(hyp_dim, 1)

    def forward(self, x):
        x = x.double() # Ensure double precision for hyperbolic ops
        feat = self.encoder(x)
        tangent = self.to_tangent(feat)
        norm = torch.norm(tangent, p=2, dim=-1, keepdim=True)
        scale = torch.clamp(norm, max=5.0) / (norm + 1e-6)
        tangent = tangent * scale
        hyp_emb = HyperbolicUtils.exp_map_zero(tangent)
        tangent_out = HyperbolicUtils.log_map_zero(hyp_emb)
        output = self.regressor(tangent_out)
        return output.squeeze()


class EuclideanRegressor(nn.Module):
    def __init__(self, input_dim, hidden_dims=[1024, 512]):
        super().__init__()
        layers = []
        in_d = input_dim
        for h_dim in hidden_dims:
            layers.append(nn.Linear(in_d, h_dim))
            layers.append(nn.BatchNorm1d(h_dim))
            layers.append(nn.ReLU())
            layers.append(nn.Dropout(0.1))
            in_d = h_dim
        layers.append(nn.Linear(in_d, 1))
        self.model = nn.Sequential(*layers)

    def forward(self, x):
        x = x.float()
        return self.model(x).squeeze()


# ==========================================
# CUSTOM LOSS: Pairwise Ranking
# ==========================================
class PairwiseRankingLoss(nn.Module):
    def __init__(self, margin=0.01):
        super().__init__()
        self.margin = margin

    def forward(self, preds, targets):
        # Assicuriamoci che siano colonne (Batch, 1)
        preds = preds.view(-1, 1)
        targets = targets.view(-1, 1)

        # Calcola le differenze a coppie (Broadcasting)
        # Matrice (B, B) dove elemento [i,j] = valore[i] - valore[j]
        pred_diff = preds - preds.t()
        target_diff = targets - targets.t()

        # Vogliamo che il segno della differenza predetta sia uguale al segno della target
        # S_ij = 1 se target[i] > target[j], -1 se <, 0 se uguale
        target_sign = torch.sign(target_diff)

        # Maschera: ignoriamo le coppie con fitness identica (non c'è ranking da imparare)
        mask = (target_sign != 0)

        # Hinge Loss: max(0, margin - (segno_vero * diff_predetta))
        # Se segno_vero * diff_predetta > margin, loss è 0 (abbiamo ordinato bene con margine)
        loss_matrix = torch.clamp(self.margin - target_sign * pred_diff, min=0)

        # Media solo sulle coppie valide
        loss = (loss_matrix * mask).sum() / (mask.sum() + 1e-8)
        return loss

# ==========================================
# HELPER: EXTRACT ALL FEATURES ONCE
# ==========================================
def extract_all_features(seq_encoder, loader, device):
    """
    Runs the sequence encoder on the provided loader and returns
    tensor of pooled features and tensor of fitness targets.
    """
    seq_encoder.eval()
    all_feats = []
    all_fitness = []

    print("Extracting features with Sequence Encoder...")
    with torch.no_grad():
        for embs, lengths, fitness, _, _ in tqdm(loader, desc="Encoding"):
            embs = embs.to(device)
            lengths = lengths.to(device)

            # Encoder outputs pooled features (B, out_dim)
            feats = seq_encoder(embs, lengths)

            all_feats.append(feats.cpu())
            all_fitness.append(fitness)

    return torch.cat(all_feats, dim=0), torch.cat(all_fitness, dim=0)

# ==========================================
# EVALUATION: Classification Focus (5 Classes)
# ==========================================
def evaluate_classification_metrics(model, loader, device, model_type="euclidean"):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for feats, fitness in loader:
            feats = feats.to(device)
            fitness = fitness.to(device)

            if model_type == "euclidean":
                preds = model(feats.float())
            else:
                preds = model(feats.double())

            all_preds.append(preds.cpu().numpy())
            all_true.append(fitness.cpu().numpy())

    y_pred_raw = np.concatenate(all_preds)
    y_true_raw = np.concatenate(all_true)

    # 1. Convertiamo i valori continui nelle 5 classi discrete
    # Usa la tua funzione helper definita all'inizio
    y_pred_cls = discretize_fitness(y_pred_raw)
    y_true_cls = discretize_fitness(y_true_raw)

    # 2. Calcolo Metriche
    # Weighted F1: Media pesata delle F1 per ogni classe (utile se le classi sono sbilanciate)
    f1 = f1_score(y_true_cls, y_pred_cls, average='weighted', zero_division=0)

    # Report dettagliato (Precision, Recall per ogni classe)
    # Lo restituiamo come stringa per stamparlo solo alla fine o quando serve
    report = classification_report(y_true_cls, y_pred_cls, zero_division=0, output_dict=True)

    return f1, report

# ==========================================
# TRAINING LOOPS (Optimized for F1 Score)
# ==========================================

def train_hyperbolic(config, train_feat_loader, val_feat_loader, feat_dim):
    print("\n--- Training Hyperbolic MLP (Target: Maximize F1 Classification) ---")
    torch.set_default_dtype(torch.float64)

    model = HyperbolicRegressor(feat_dim, [512], config.hyp_dim).to(config.device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=config.hyp_lr)

    # NOTA: Usiamo MSELoss per addestrare i valori corretti rispetto alle soglie
    criterion = nn.HuberLoss()

    best_f1 = -1.0
    best_rho = -1.0
    best_report = None

    for epoch in range(config.epochs_trans):
        model.train()
        total_loss = 0.0

        for feats, fitness in train_feat_loader:
            feats = feats.to(config.device).double()
            fitness = fitness.to(config.device).double()

            pred = model(feats)
            loss = criterion(pred, fitness)

            optimizer.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            total_loss += loss.item()

        train_f1, _ = evaluate_classification_metrics(model, train_feat_loader, config.device, "hyperbolic")
        # Valutazione
        val_rho, val_mse, _ = evaluate_proteingym_metrics(model, val_feat_loader, config.device, "hyperbolic")

        # Save based on Spearman instead of F1
        
        val_f1, val_report = evaluate_classification_metrics(model, val_feat_loader, config.device, "hyperbolic")

        print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_feat_loader):.4f} | Train F1 (Weighted): {train_f1:.4f}| Val F1 (Weighted): {val_f1:.4f}| Val Spearman: {val_rho:.4f} | Val MSE: {val_mse:.4f}")

        # Salviamo il modello se F1 migliora
        if val_rho > best_rho:
            best_rho = val_rho
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_report = val_report

    return best_rho, best_f1, best_report, model

def train_euclidean(config, train_feat_loader, val_feat_loader, feat_dim):
    print("\n--- Training Euclidean MLP (Target: Maximize F1 Classification) ---")
    torch.set_default_dtype(torch.float32)

    model = EuclideanRegressor(feat_dim, config.euc_hidden).to(config.device)
    optimizer = torch.optim.Adam(model.parameters(), lr=config.euc_lr)

    criterion = nn.HuberLoss()

    best_f1 = -1.0
    best_rho = -1.0
    best_report = None

    for epoch in range(config.epochs_cis):
        model.train()
        total_loss = 0.0

        for feats, fitness in train_feat_loader:
            feats = feats.to(config.device).float()
            fitness = fitness.to(config.device).float()

            pred = model(feats)
            loss = criterion(pred, fitness)

            optimizer.zero_grad()
            loss.backward()
            optimizer.step()
            total_loss += loss.item()

        train_f1, _ = evaluate_classification_metrics(model, train_feat_loader, config.device, "euclidean")
        val_rho, val_mse, _ = evaluate_proteingym_metrics(model, val_feat_loader, config.device, "euclidean")
        val_f1, val_report = evaluate_classification_metrics(model, val_feat_loader, config.device, "euclidean")

        print(f"Epoch {epoch+1} | Loss: {total_loss / len(train_feat_loader):.4f} | Train F1 (Weighted): {train_f1:.4f} | Val F1 (Weighted): {val_f1:.4f}| Val Spearman: {val_rho:.4f} | Val MSE: {val_mse:.4f}")

        if val_rho > best_rho:
            best_rho = val_rho
        if val_f1 > best_f1:
            best_f1 = val_f1
            best_report = val_report

    return best_rho, best_f1, best_report, model

from sklearn.metrics import ndcg_score

def evaluate_proteingym_metrics(model, loader, device, model_type):
    model.eval()
    all_preds = []
    all_true = []

    with torch.no_grad():
        for feats, fitness in loader:
            feats = feats.to(device)
            fitness = fitness.to(device)

            if model_type == "euclidean":
                preds = model(feats.float())
            else:
                preds = model(feats.double())

            all_preds.append(preds.cpu().numpy())
            all_true.append(fitness.cpu().numpy())

    y_pred_raw = np.concatenate(all_preds)
    y_true_raw = np.concatenate(all_true)

    # 1. Spearman's Correlation (Primary Metric)
    # No changes needed here, Spearman handles negatives fine
    rho, _ = spearmanr(y_true_raw, y_pred_raw)

    # 2. MSE
    mse = mean_squared_error(y_true_raw, y_pred_raw)

    # 3. NDCG (Fix for Negative Values)
    # We shift the "True" fitness values so the lowest value becomes 0.
    # This keeps the relative ranking identical but makes all values non-negative for sklearn.
    y_true_shifted = y_true_raw - y_true_raw.min()
    
    # If values are still 0 (e.g. constant array), add a tiny epsilon to avoid errors, 
    # though strict 0 is usually fine for NDCG as "irrelevant".
    
    # NDCG expects shape (1, n_samples) for a single ranking list
    ndcg = ndcg_score(
        np.expand_dims(y_true_shifted, axis=0), 
        np.expand_dims(y_pred_raw, axis=0)
    )

    return rho, mse, ndcg

In [ ]:


# --- MAIN ---
def main_func():
    # 1. FIX CRITICO: Ripristina il default a float32 per evitare conflitti con esecuzioni passate
    torch.set_default_dtype(torch.float32)

    if not os.path.exists(Config.data_path):
        print(f"File not found: {Config.data_path}")
        return

    # 1. Data Loading (Raw Sequences)
    dataset = MutationDataset(
        Config.data_path,
        max_samples=Config.max_samples,
        max_seq_len=Config.max_seq_len
    )

    train_size = int((1 - Config.val_frac) * len(dataset))
    val_size = len(dataset) - train_size
    train_ds, val_ds = torch.utils.data.random_split(
        dataset, [train_size, val_size],
        generator=torch.Generator().manual_seed(Config.seed)
    )

    # Initial Loaders for feature extraction
    train_loader_seq = DataLoader(train_ds, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)
    val_loader_seq = DataLoader(val_ds, batch_size=Config.batch_size, shuffle=False, collate_fn=collate_fn)

    print(f"Dataset Loaded. Seq feature dim (max_w): {dataset.max_w}, seq len (max_h): {dataset.max_h}")

    # 2. Initialize Encoder ONCE
    print("\n--- Initializing Sequence Encoder ---")

    seq_enc = SequenceEncoder(
        input_dim=dataset.max_w,
        hidden_size=Config.seq_hidden,
        num_layers=Config.seq_layers,
        dropout=Config.seq_dropout,
        nhead=Config.n_head  # Assicurati che questo sia nella tua Config
    ).to(Config.device).float()  # <--- AGGIUNTO .float() PER SICUREZZA

    # 3. Extract Features (Run Encoder once)
    print("Pre-computing features for Training set...")
    # Assicuriamo che l'encoder sia in float32 prima di usarlo
    seq_enc.float()
    train_feats, train_fitness = extract_all_features(seq_enc, train_loader_seq, Config.device)

    print("Pre-computing features for Validation set...")
    val_feats, val_fitness = extract_all_features(seq_enc, val_loader_seq, Config.device)

    feat_dim = seq_enc.out_dim
    print(f"Feature Extraction Complete. Dimension: {feat_dim}")

    # 4. Create Feature Loaders (TensorDataset is faster for pre-computed data)
    train_feat_ds = TensorDataset(train_feats, train_fitness)
    val_feat_ds = TensorDataset(val_feats, val_fitness)

    # Shuffle only train
    train_feat_loader = DataLoader(train_feat_ds, batch_size=Config.batch_size, shuffle=True)
    val_feat_loader = DataLoader(val_feat_ds, batch_size=Config.batch_size, shuffle=False)

    # 5. Train Hyperbolic MLP
    # Nota: ora catturiamo anche 'hyp_model'
    hyp_sperm, hyp_f1, hyp_rep, hyp_model = train_hyperbolic(Config, train_feat_loader, val_feat_loader, feat_dim)

    # 6. Train Euclidean MLP
    torch.set_default_dtype(torch.float32)
    euc_sperm, euc_f1, euc_rep, euc_model = train_euclidean(Config, train_feat_loader, val_feat_loader, feat_dim)

    print("\n================ FINAL COMPARISON (CLASSIFICATION F1) ================")
    print(f"{'Metric':<10} | {'Hyperbolic':<15} | {'Euclidean':<15}")
    print("-" * 45)
    print(f"{'F1 Score':<10} | {hyp_f1:<15.4f} | {euc_f1:<15.4f}")
    print("-" * 45)
    print(f"{'Sperm coeff':<10} | {hyp_sperm:<15.4f} | {euc_sperm:<15.4f}")

    print("\n--- Generating Confusion Matrices ---")

    # Matrice Iperbolica
    visualize_confusion_matrix(hyp_model, val_feat_loader, Config.device,
                               model_type="hyperbolic", title="Hyperbolic Results")

    # Matrice Euclidea
    visualize_confusion_matrix(euc_model, val_feat_loader, Config.device,
                               model_type="euclidean", title="Euclidean Results")

    print("\n--- Generating Fitness Distribution Plots ---")

    # Plot per modello Iperbolico
    visualize_fitness_distribution_colored(hyp_model, val_feat_loader, Config.device, "hyperbolic")

    # Plot per modello Euclideo
    visualize_fitness_distribution_colored(euc_model, val_feat_loader, Config.device, "euclidean")

if __name__ == "__main__":
    main_func()